In [1]:
import os
import sys
import io

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, '.')
from models import MLP

os.makedirs('gifs', exist_ok=True)

In [2]:
LABEL_MAP = {'red': 0, 'blue': 1}
COLORS = {0: 'red', 1: 'blue'}

def make_frame(model, X_np, y_np, epoch, loss, acc):
    fig, ax = plt.subplots(figsize=(6, 5))

    margin = 0.5
    x_min, x_max = X_np[:, 0].min() - margin, X_np[:, 0].max() + margin
    y_min, y_max = X_np[:, 1].min() - margin, X_np[:, 1].max() + margin

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        Z = torch.softmax(model(grid), dim=1)[:, 1].numpy().reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=100, cmap='RdBu', alpha=0.75, vmin=0, vmax=1)
    point_colors = [COLORS[int(l)] for l in y_np]
    ax.scatter(X_np[:, 0], X_np[:, 1], c=point_colors,
               edgecolors='k', linewidths=0.5, s=30, zorder=5)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_title(f'Epoch {epoch}  |  loss={loss:.4f}  |  acc={acc:.3f}')

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=80, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()

In [3]:
DATASET_PATHS = [
    'datasets/dataset1_two_circles.csv',
    'datasets/dataset2_three_circles.csv',
    'datasets/dataset3_stripes.csv',
    'datasets/dataset4_sine.csv',
    'datasets/dataset5_moons.csv',
]

EPOCHS = 1000
LR = 0.1
CAPTURE_EVERY = 10   # one GIF frame per N epochs
GIF_DURATION = 50    # ms per frame

for path in DATASET_PATHS:
    name = os.path.splitext(os.path.basename(path))[0]
    print(f"\n=== {name} ===")

    df = pd.read_csv(path)
    X_np = df[['x', 'y']].values.astype(np.float32)
    y_np = df['label'].map(LABEL_MAP).values.astype(np.int64)
    X = torch.tensor(X_np)
    y = torch.tensor(y_np)

    model = MLP(hidden_dims=[5])
    optimizer = torch.optim.SGD(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    frames = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()

        if epoch == 1 or epoch % CAPTURE_EVERY == 0:
            with torch.no_grad():
                acc = (model(X).argmax(dim=1) == y).float().mean().item()
            frames.append(make_frame(model, X_np, y_np, epoch, loss.item(), acc))

        if epoch % 200 == 0:
            with torch.no_grad():
                acc = (model(X).argmax(dim=1) == y).float().mean().item()
            print(f"  epoch {epoch}/{EPOCHS}  loss={loss.item():.4f}  acc={acc:.4f}")

    gif_path = f'gifs/{name}.gif'
    frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        loop=0,
        duration=GIF_DURATION,
    )
    print(f"  saved -> {gif_path}")


=== dataset1_two_circles ===
  epoch 200/1000  loss=0.4580  acc=0.9050
  epoch 400/1000  loss=0.1324  acc=1.0000
  epoch 600/1000  loss=0.0508  acc=1.0000
  epoch 800/1000  loss=0.0281  acc=1.0000
  epoch 1000/1000  loss=0.0186  acc=1.0000
  saved -> gifs/dataset1_two_circles.gif

=== dataset2_three_circles ===
  epoch 200/1000  loss=0.6298  acc=0.6667
  epoch 400/1000  loss=0.6226  acc=0.6667
  epoch 600/1000  loss=0.6138  acc=0.6667
  epoch 800/1000  loss=0.6018  acc=0.6667
  epoch 1000/1000  loss=0.5846  acc=0.6700
  saved -> gifs/dataset2_three_circles.gif

=== dataset3_stripes ===
  epoch 200/1000  loss=0.4392  acc=0.6730
  epoch 400/1000  loss=0.3442  acc=0.9115
  epoch 600/1000  loss=0.2153  acc=0.9990
  epoch 800/1000  loss=0.1390  acc=1.0000
  epoch 1000/1000  loss=0.0983  acc=1.0000
  saved -> gifs/dataset3_stripes.gif

=== dataset4_sine ===
  epoch 200/1000  loss=0.3938  acc=0.7950
  epoch 400/1000  loss=0.3905  acc=0.7960
  epoch 600/1000  loss=0.3891  acc=0.7965
  epoch 8